In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

Fuente: https://www.kaggle.com/datasets/mirbektoktogaraev/madrid-real-estate-market

Se carga la información

In [4]:
file_df = pd.read_csv("data_source/madrid_real_estate_market.csv", encoding="utf-8")
temp_df, test_df = train_test_split(file_df, test_size=0.2, random_state=42, shuffle=True)

train_df, val_df = train_test_split(temp_df, test_size=0.25, shuffle=True)

train_df.head()

,Unnamed: 0,id,title,subtitle,sq_mt_built,sq_mt_useful,n_rooms,n_bathrooms,n_floors,sq_mt_allotment,...,energy_certificate,has_parking,has_private_parking,has_public_parking,is_parking_included_in_price,parking_price,is_orientation_north,is_orientation_west,is_orientation_south,is_orientation_east
4078,4078,17664,"Piso en venta en paseo de la Reina Cristina, 17","Jerónimos, Madrid",153.0,125.0,3,2.0,NaN,NaN,...,no indicado,True,NaN,NaN,True,0.0,True,False,True,False
3471,3471,18271,Piso en venta en calle de Lope de Haro s/n,"Berruguete, Madrid",75.0,NaN,3,2.0,NaN,NaN,...,en trámite,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11246,11246,10496,Piso en venta en calle de Caracas,"Almagro, Madrid",70.0,NaN,2,1.0,NaN,NaN,...,no indicado,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7189,7189,14553,Chalet pareado en venta en calle de Gerda Taro,"El Plantío, Madrid",270.0,264.0,6,5.0,4.0,280.0,...,en trámite,True,NaN,NaN,True,0.0,False,True,True,True
19360,19360,2382,Piso en venta en calle Ardemans,"Guindalera, Madrid",54.0,49.0,2,1.0,NaN,NaN,...,en trámite,False,NaN,NaN,NaN,NaN,False,True,False,False


Se extrae información de la columna "ne'neighborhood_id' para extraer el precio por m2 y el distrito

In [5]:
# Extraer el valor por m2
train_df['Precio_por_m2'] = train_df['neighborhood_id'].str.extract(r'\(([\d.]+) €/m2\)')[0].astype(float)
val_df['Precio_por_m2'] = val_df['neighborhood_id'].str.extract(r'\(([\d.]+) €/m2\)')[0].astype(float)
test_df['Precio_por_m2'] = test_df['neighborhood_id'].str.extract(r'\(([\d.]+) €/m2\)')[0].astype(float)

train_df['Distrito'] = train_df['neighborhood_id'].str.extract(r'District \d+: (.+)')
val_df['Distrito'] = val_df['neighborhood_id'].str.extract(r'District \d+: (.+)')
test_df['Distrito'] = test_df['neighborhood_id'].str.extract(r'District \d+: (.+)')
# end

Se chequea la información de las columnas y sus formatos de valores. Igualmente se verifican a si hay valores nulos.

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 13044 entries, 4078 to 4100
Data columns (total 60 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    13044 non-null  int64  
 1   id                            13044 non-null  int64  
 2   title                         13044 non-null  str    
 3   subtitle                      13044 non-null  str    
 4   sq_mt_built                   12970 non-null  float64
 5   sq_mt_useful                  4934 non-null   float64
 6   n_rooms                       13044 non-null  int64  
 7   n_bathrooms                   13036 non-null  float64
 8   n_floors                      843 non-null    float64
 9   sq_mt_allotment               836 non-null    float64
 10  latitude                      0 non-null      float64
 11  longitude                     0 non-null      float64
 12  raw_address                   9797 non-null   str    
 13  is_exact_addres

In [7]:
# Función
columns_to_drop = [
    "Unnamed: 0", # No aporta información relevante
    "id", # No aporta información relevante
    "title", # No aporta información relevante
    "operation", # tiene un único valor
    "raw_address", # No aporta información relevante
    'is_exact_address_hidden', # No aporta información relevante
    # 'street_name', # No aporta información relevante
    'neighborhood_id', # No aporta información relevante
    'is_rent_price_known', # No aporta información relevante
    'is_buy_price_known', # No aporta información relevante
    # 'is_orientation_north', # No aporta información relevante
    # 'is_orientation_west', # No aporta información relevante
    # 'is_orientation_south', # No aporta información relevante
    # 'is_orientation_east' # No aporta información relevante
]
train_df.drop(columns_to_drop, inplace=True, axis=1)
val_df.drop(columns_to_drop, inplace=True, axis=1)
test_df.drop(columns_to_drop, inplace=True, axis=1)
# end

print(f"Columnas restantes: {(train_df.columns.size)}")

Columnas restantes: 51


Se determina que columnas poseen 70% de valores nulos o superior y se descartan por no tener la información disponible y por no considerarse viable rellenar con valores arbitrarios.

In [8]:
null_tolerance = 0.25

columns_to_drop_2 = []
for column in train_df.columns.to_list():
    if (train_df[column].isnull().sum() / len(train_df)) >= null_tolerance:
        columns_to_drop_2.append(str(column))
print(columns_to_drop_2)
print(f"Número de campos a descartar: {len(columns_to_drop_2)}")

train_df.drop(columns_to_drop_2, inplace=True, axis=1)

['sq_mt_useful', 'n_floors', 'sq_mt_allotment', 'latitude', 'longitude', 'street_name', 'street_number', 'portal', 'door', 'rent_price_by_area', 'built_year', 'has_central_heating', 'has_individual_heating', 'are_pets_allowed', 'has_ac', 'has_fitted_wardrobes', 'has_garden', 'has_pool', 'has_terrace', 'has_balcony', 'has_storage_room', 'is_furnished', 'is_kitchen_equipped', 'is_accessible', 'has_green_zones', 'has_private_parking', 'has_public_parking', 'is_parking_included_in_price', 'parking_price', 'is_orientation_north', 'is_orientation_west', 'is_orientation_south', 'is_orientation_east']
Número de campos a descartar: 33


In [9]:
columns_to_drop_2 = []
for column in val_df.columns.to_list():
    if (val_df[column].isnull().sum() / len(val_df)) >= null_tolerance:
        columns_to_drop_2.append(str(column))
print(columns_to_drop_2)
print(f"Número de campos a descartar: {len(columns_to_drop_2)}")

val_df.drop(columns_to_drop_2, inplace=True, axis=1)

['sq_mt_useful', 'n_floors', 'sq_mt_allotment', 'latitude', 'longitude', 'street_name', 'street_number', 'portal', 'door', 'rent_price_by_area', 'built_year', 'has_central_heating', 'has_individual_heating', 'are_pets_allowed', 'has_ac', 'has_fitted_wardrobes', 'has_garden', 'has_pool', 'has_terrace', 'has_balcony', 'has_storage_room', 'is_furnished', 'is_kitchen_equipped', 'is_accessible', 'has_green_zones', 'has_private_parking', 'has_public_parking', 'is_parking_included_in_price', 'parking_price', 'is_orientation_north', 'is_orientation_west', 'is_orientation_south', 'is_orientation_east']
Número de campos a descartar: 33


In [10]:
columns_to_drop_2 = []
for column in test_df.columns.to_list():
    if (test_df[column].isnull().sum() / len(test_df)) >= null_tolerance:
        columns_to_drop_2.append(str(column))
print(columns_to_drop_2)
print(f"Número de campos a descartar: {len(columns_to_drop_2)}")

test_df.drop(columns_to_drop_2, inplace=True, axis=1)

['sq_mt_useful', 'n_floors', 'sq_mt_allotment', 'latitude', 'longitude', 'street_name', 'street_number', 'portal', 'door', 'rent_price_by_area', 'built_year', 'has_central_heating', 'has_individual_heating', 'are_pets_allowed', 'has_ac', 'has_fitted_wardrobes', 'has_garden', 'has_pool', 'has_terrace', 'has_balcony', 'has_storage_room', 'is_furnished', 'is_kitchen_equipped', 'is_accessible', 'has_green_zones', 'has_private_parking', 'has_public_parking', 'is_parking_included_in_price', 'parking_price', 'is_orientation_north', 'is_orientation_west', 'is_orientation_south', 'is_orientation_east']
Número de campos a descartar: 33


Se elimina la palabra expresión *", Madrid"* de todos los registros de la columna subtitle por resultar redundante. (Todos los registros por usar se refieren a Madrid por naturaleza del dataset). 

In [11]:
# Función
train_df["subtitle"] = train_df["subtitle"].apply(lambda x: x.replace(", Madrid", ""))
val_df["subtitle"] = val_df["subtitle"].apply(lambda x: x.replace(", Madrid", ""))
test_df["subtitle"] = test_df["subtitle"].apply(lambda x: x.replace(", Madrid", ""))
# end

train_df["subtitle"]

4078           Jerónimos
3471          Berruguete
11246            Almagro
7189          El Plantío
19360         Guindalera
              ...       
20680    Palos de Moguer
2574       Bellas Vistas
15632          Chamartín
10584          Hortaleza
4100              Retiro
Name: subtitle, Length: 13044, dtype: str

Se revisa la lista de columnas y su cantidad.

In [12]:
print(train_df.columns)
print(f"Número de columnas: {len(train_df.columns)}")

Index(['subtitle', 'sq_mt_built', 'n_rooms', 'n_bathrooms', 'floor',
       'is_floor_under', 'rent_price', 'buy_price', 'buy_price_by_area',
       'house_type_id', 'is_renewal_needed', 'is_new_development', 'has_lift',
       'is_exterior', 'energy_certificate', 'has_parking', 'Precio_por_m2',
       'Distrito'],
      dtype='str')
Número de columnas: 18


Se Eliminan columnas no relevantes tomando en consideración que se usará un modelo de ML/DP

## Verificación de columnas

In [13]:
total_row_number = len(train_df)
columns_wit_na = []
for column in train_df.columns:
    train_df[column].isna().sum() != total_row_number
    columns_wit_na.append(column)
print(columns_wit_na)
print(len(columns_wit_na))

['subtitle', 'sq_mt_built', 'n_rooms', 'n_bathrooms', 'floor', 'is_floor_under', 'rent_price', 'buy_price', 'buy_price_by_area', 'house_type_id', 'is_renewal_needed', 'is_new_development', 'has_lift', 'is_exterior', 'energy_certificate', 'has_parking', 'Precio_por_m2', 'Distrito']
18


Se eliminan registros que tengan un determinado porcentaje de valores nulos.

In [14]:
conteo_inicial_data = len(train_df)

# Función
NULL_VALUES_PORCENTAGE_PER_ROW = 0.4
train_df.dropna(thresh=(1-NULL_VALUES_PORCENTAGE_PER_ROW) * len(train_df.columns), inplace=True)
val_df.dropna(thresh=(1-NULL_VALUES_PORCENTAGE_PER_ROW) * len(val_df.columns), inplace=True)
test_df.dropna(thresh=(1-NULL_VALUES_PORCENTAGE_PER_ROW) * len(test_df.columns), inplace=True)
# end

conteo_actualizado_data = len(train_df)
print(f"Se eliminaron: {conteo_inicial_data - conteo_actualizado_data} registros.")

Se eliminaron: 0 registros.


## energy_certificate

Se realizan conteo de valores de "energy_certificate"

In [15]:
print(train_df["energy_certificate"].value_counts(), f"\nNúmero de valores: {len(train_df["energy_certificate"])}")

energy_certificate
en trámite         6602
no indicado        2226
E                  1572
D                   669
G                   540
F                   421
A                   351
C                   346
B                   259
inmueble exento      58
Name: count, dtype: int64 
Número de valores: 13044


Se cambian los valores diferentes a una letra (en trámite o no indicado) en un valor igual a la letra "O".

In [16]:
# Función
train_df.loc[~train_df["energy_certificate"].isin(["A", "B", "C", "D", "E", "F", "G"]), "energy_certificate"] = "other"
val_df.loc[~val_df["energy_certificate"].isin(["A", "B", "C", "D", "E", "F", "G"]), "energy_certificate"] = "other"
test_df.loc[~test_df["energy_certificate"].isin(["A", "B", "C", "D", "E", "F", "G"]), "energy_certificate"] = "other"
# end

train_df["energy_certificate"].value_counts()

energy_certificate
other    8886
E        1572
D         669
G         540
F         421
A         351
C         346
B         259
Name: count, dtype: int64

## floor

Se realiza conteo de de valores de "floor"

In [17]:
print(train_df["floor"].value_counts())
print(len(train_df["floor"]))

floor
1                       2705
2                       2118
3                       1770
4                       1384
Bajo                    1297
5                        782
6                        567
7                        356
8                        192
Entreplanta exterior     143
9                        106
Semi-sótano exterior      34
Semi-sótano interior      18
Entreplanta interior      17
Sótano interior           14
Sótano                     5
Entreplanta                2
Sótano exterior            1
Name: count, dtype: int64
13044


Se sustituyen los valores no numéricos por valores numéricos pero que seguirán siendo string para efectos de la creación del archivo. Todo "bajo" será igual a "0" y caulquier otro valor no numérico será igual a "-1".

In [18]:
# Función
# Cambiar valores de "floor" a 0 si es "Bajo"
train_df.loc[train_df["floor"].str.lower() == "bajo", "floor"] = "0"
val_df.loc[val_df["floor"].str.lower() == "bajo", "floor"] = "0"
test_df.loc[test_df["floor"].str.lower() == "bajo", "floor"] = "0"

# Cambiar valores de "floor" a -1 si no es un número entero u otro tipo de valor
train_df.loc[(train_df["floor"].str.isdigit() == False) | (train_df["floor"].isna()), "floor"] = "-1"
val_df.loc[(val_df["floor"].str.isdigit() == False) | (val_df["floor"].isna()), "floor"] = "-1"
test_df.loc[(test_df["floor"].str.isdigit() == False) | (test_df["floor"].isna()), "floor"] = "-1"
# end

print(train_df["floor"].value_counts())
print(len(train_df["floor"]))

floor
1     2705
2     2118
3     1770
-1    1767
4     1384
0     1297
5      782
6      567
7      356
8      192
9      106
Name: count, dtype: int64
13044


Se chequea si existen campos o columnas con valores nulos.

In [19]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 13044 entries, 4078 to 4100
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subtitle            13044 non-null  str    
 1   sq_mt_built         12970 non-null  float64
 2   n_rooms             13044 non-null  int64  
 3   n_bathrooms         13036 non-null  float64
 4   floor               13044 non-null  str    
 5   is_floor_under      12354 non-null  object 
 6   rent_price          13044 non-null  int64  
 7   buy_price           13044 non-null  int64  
 8   buy_price_by_area   13044 non-null  int64  
 9   house_type_id       12812 non-null  str    
 10  is_renewal_needed   13044 non-null  bool   
 11  is_new_development  12438 non-null  object 
 12  has_lift            11642 non-null  object 
 13  is_exterior         11250 non-null  object 
 14  energy_certificate  13044 non-null  str    
 15  has_parking         13044 non-null  bool   
 16  Precio_por_m2     

In [20]:
# Función
# Se exluyen todas las filas que no contengan valores en los target
target = ["buy_price", "rent_price"]
train_df = train_df.dropna(subset=target, axis=0)
val_df = val_df.dropna(subset=target, axis=0)
test_df = test_df.dropna(subset=target, axis=0)
# end

Se rellenan los datos faltantes con la moda de cada variable.

In [21]:
for column in train_df.columns:
    stats = train_df[column].mode()[0]
    train_df[column] = train_df[column].fillna(stats)
    val_df[column] = val_df[column].fillna(stats)
    test_df[column] = test_df[column].fillna(stats)

train_df.info()

<class 'pandas.DataFrame'>
Index: 13044 entries, 4078 to 4100
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subtitle            13044 non-null  str    
 1   sq_mt_built         13044 non-null  float64
 2   n_rooms             13044 non-null  int64  
 3   n_bathrooms         13044 non-null  float64
 4   floor               13044 non-null  str    
 5   is_floor_under      13044 non-null  object 
 6   rent_price          13044 non-null  int64  
 7   buy_price           13044 non-null  int64  
 8   buy_price_by_area   13044 non-null  int64  
 9   house_type_id       13044 non-null  str    
 10  is_renewal_needed   13044 non-null  bool   
 11  is_new_development  13044 non-null  object 
 12  has_lift            13044 non-null  object 
 13  is_exterior         13044 non-null  object 
 14  energy_certificate  13044 non-null  str    
 15  has_parking         13044 non-null  bool   
 16  Precio_por_m2     

In [22]:
train_df["house_type_id"].value_counts()

house_type_id
HouseType 1: Pisos            10860
HouseType 2: Casa o chalet     1134
HouseType 5: Áticos             640
HouseType 4: Dúplex             410
Name: count, dtype: int64

In [23]:
train_df.head()

,subtitle,sq_mt_built,n_rooms,n_bathrooms,floor,is_floor_under,rent_price,buy_price,buy_price_by_area,house_type_id,is_renewal_needed,is_new_development,has_lift,is_exterior,energy_certificate,has_parking,Precio_por_m2,Distrito
4078,Jerónimos,153.0,3,2.0,1,False,1795,556000,3634,HouseType 1: Pisos,False,False,True,True,other,True,6739.32,Retiro
3471,Berruguete,75.0,3,2.0,6,False,1248,319000,4253,HouseType 1: Pisos,True,False,True,True,other,False,3273.56,Tetuán
11246,Almagro,70.0,2,1.0,1,False,1644,490000,7000,HouseType 1: Pisos,False,False,False,False,other,False,6564.27,Chamberí
7189,El Plantío,270.0,6,5.0,-1,False,1869,588000,2178,HouseType 2: Casa o chalet,True,False,True,True,other,True,2569.96,Moncloa
19360,Guindalera,54.0,2,1.0,7,False,1230,312000,5778,HouseType 1: Pisos,False,False,True,True,other,False,4367.90,Salamanca


Se elimina la palabra "HouseType X: " de los registros de la columna na "house_type_id".

In [24]:
# Función
train_df['house_type_id'] = train_df['house_type_id'].str.replace(r'^HouseType \d+: ', '', regex=True)
val_df['house_type_id'] = val_df['house_type_id'].str.replace(r'^HouseType \d+: ', '', regex=True)
test_df['house_type_id'] = test_df['house_type_id'].str.replace(r'^HouseType \d+: ', '', regex=True)
# end

train_df.head()

,subtitle,sq_mt_built,n_rooms,n_bathrooms,floor,is_floor_under,rent_price,buy_price,buy_price_by_area,house_type_id,is_renewal_needed,is_new_development,has_lift,is_exterior,energy_certificate,has_parking,Precio_por_m2,Distrito
4078,Jerónimos,153.0,3,2.0,1,False,1795,556000,3634,Pisos,False,False,True,True,other,True,6739.32,Retiro
3471,Berruguete,75.0,3,2.0,6,False,1248,319000,4253,Pisos,True,False,True,True,other,False,3273.56,Tetuán
11246,Almagro,70.0,2,1.0,1,False,1644,490000,7000,Pisos,False,False,False,False,other,False,6564.27,Chamberí
7189,El Plantío,270.0,6,5.0,-1,False,1869,588000,2178,Casa o chalet,True,False,True,True,other,True,2569.96,Moncloa
19360,Guindalera,54.0,2,1.0,7,False,1230,312000,5778,Pisos,False,False,True,True,other,False,4367.90,Salamanca


In [25]:
train_df["subtitle"].value_counts()

subtitle
Chamartín                           536
Moncloa                             449
Chamberí                            363
Centro                              317
Hortaleza                           286
                                   ... 
El Pardo                              6
Barajas                               5
Campo de las Naciones-Corralejos      3
Timón                                 2
Casco Histórico de Barajas            2
Name: count, Length: 145, dtype: int64

In [26]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 13044 entries, 4078 to 4100
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subtitle            13044 non-null  str    
 1   sq_mt_built         13044 non-null  float64
 2   n_rooms             13044 non-null  int64  
 3   n_bathrooms         13044 non-null  float64
 4   floor               13044 non-null  str    
 5   is_floor_under      13044 non-null  object 
 6   rent_price          13044 non-null  int64  
 7   buy_price           13044 non-null  int64  
 8   buy_price_by_area   13044 non-null  int64  
 9   house_type_id       13044 non-null  str    
 10  is_renewal_needed   13044 non-null  bool   
 11  is_new_development  13044 non-null  object 
 12  has_lift            13044 non-null  object 
 13  is_exterior         13044 non-null  object 
 14  energy_certificate  13044 non-null  str    
 15  has_parking         13044 non-null  bool   
 16  Precio_por_m2     

Se reaiza prueba de función de limpieza de datos de validación y test.

In [27]:
val_df["n_bathrooms"].value_counts()

n_bathrooms
1.0     1862
2.0     1463
3.0      462
4.0      243
5.0      182
6.0       81
7.0       32
8.0       15
9.0        4
14.0       2
10.0       1
12.0       1
13.0       1
Name: count, dtype: int64

In [28]:
print(test_df.info())
print(val_df.info())

<class 'pandas.DataFrame'>
Index: 4349 entries, 21257 to 16006
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subtitle            4349 non-null   str    
 1   sq_mt_built         4349 non-null   float64
 2   n_rooms             4349 non-null   int64  
 3   n_bathrooms         4349 non-null   float64
 4   floor               4349 non-null   str    
 5   is_floor_under      4349 non-null   object 
 6   rent_price          4349 non-null   int64  
 7   buy_price           4349 non-null   int64  
 8   buy_price_by_area   4349 non-null   int64  
 9   house_type_id       4349 non-null   str    
 10  is_renewal_needed   4349 non-null   bool   
 11  is_new_development  4349 non-null   object 
 12  has_lift            4349 non-null   object 
 13  is_exterior         4349 non-null   object 
 14  energy_certificate  4349 non-null   str    
 15  has_parking         4349 non-null   bool   
 16  Precio_por_m2    

Se exporta a archivos los datos limpios para su uso posterior.

In [ ]:
train_df.to_csv("03_datasets/train_dataset_base.csv", index=False)
val_df.to_csv("03_datasets/val_dataset_base.csv", index=False)
test_df.to_csv("03_datasets/test_dataset_base.csv", index=False)